In [1]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_community.document_loaders import DirectoryLoader

## load all the pdf files from the directory
def document_read(path):
    dir_loader=DirectoryLoader(path,
        glob="**/*.pdf", ## Pattern to match files  
        loader_cls= PyMuPDFLoader, ##loader class to use
        show_progress=False)
    pdf_documents=dir_loader.load()
    return pdf_documents

path = "../RAG LLM document to response/Document"
pdf_documents = document_read(path)
# pdf_documents

C:\Users\prati\AppData\Local\Temp\ipykernel_16808\3208682074.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [2]:
# It will extract and save all the image into the image folder 
import fitz  # PyMuPDF
import os

def extract_images_from_pdf(pdf_path, output_dir):
    pdf = fitz.open(pdf_path)

    os.makedirs(output_dir, exist_ok=True)

    for page_num in range(len(pdf)):
        page = pdf[page_num]

        for img_index, img in enumerate(page.get_images(full=True)):
            xref = img[0]

            base_image = pdf.extract_image(xref)
            image_bytes = base_image["image"]
            image_ext = base_image["ext"]

            image_name = f"page_{page_num+1}_{img_index}.{image_ext}"

            with open(os.path.join(output_dir, image_name),"wb") as f:
                f.write(image_bytes)

    pdf.close()

In [3]:
img = extract_images_from_pdf("../RAG LLM document to response/Document/Practical Statistics for Data Scientists.pdf",
                              "../RAG LLM document to response/images")

In [4]:
# Now extracting the tables and carts in the csv file

import camelot

def Extract_table(doc_path,csv_path):
    tables = camelot.read_pdf(doc_path, pages="all")

    for i, table in enumerate(tables):
        df = table.df
        df.to_csv(f"{csv_path}table_{i}.csv", index=False)

doc_path = "../RAG LLM document to response/Document/Practical Statistics for Data Scientists.pdf"
csv_path =  "../RAG LLM document to response/Tables/"

table = Extract_table(doc_path,csv_path)

MediaBox missing from Page id 1 (and not inherited), defaulting to US Letter (612x792)
MediaBox missing from Page id 5 (and not inherited), defaulting to US Letter (612x792)
MediaBox missing from Page id 23 (and not inherited), defaulting to US Letter (612x792)
MediaBox missing from Page id 32 (and not inherited), defaulting to US Letter (612x792)
MediaBox missing from Page id 35 (and not inherited), defaulting to US Letter (612x792)
MediaBox missing from Page id 37 (and not inherited), defaulting to US Letter (612x792)
MediaBox missing from Page id 39 (and not inherited), defaulting to US Letter (612x792)
MediaBox missing from Page id 41 (and not inherited), defaulting to US Letter (612x792)
MediaBox missing from Page id 55 (and not inherited), defaulting to US Letter (612x792)
MediaBox missing from Page id 59 (and not inherited), defaulting to US Letter (612x792)
MediaBox missing from Page id 68 (and not inherited), defaulting to US Letter (612x792)
MediaBox missing from Page id 77 (

In [ ]:
# Now converting all the image into the vector embedding using the ollama --> llava multimodel

import ollama

def Image_text(img_path):
    response = ollama.chat(model="llava:7b",
        messages=[{"role": "user",
            "content": "Describe this image in detail of 100 words.",
            "images": [img_path]}])

    caption = response["message"]["content"]
    return caption
    
img_path = "../RAG LLM document to response/images/page_114_0.jpeg"
print(Image_text(img_path))

In [5]:
# Text splitter using the default RecursiveCharacterTextSplitter with overlap of 10-20% of token size
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
    
    return split_docs

chuncks = split_documents(pdf_documents,chunk_size=1000,chunk_overlap=200)

Split 562 documents into 853 chunks

Example chunk:
Content: www.allitebooks.com...


In [6]:
# Embedding and vectorstoreDB
import numpy as np
from sentence_transformers import SentenceTransformer
import ollama 

# ---------------------------------------------------
# Load Embedding Model + Generate Embeddings
# ---------------------------------------------------
def generate_embeddings(model_name, documents):

    # Extract text from Document objects
    embed_list = list()
    for doc in documents:
        text = doc.page_content
        embeddings = ollama.embed(model_name, text)
        embed_list.append(embeddings.embeddings)

    return embed_list


# Model Name
model_name = "embeddinggemma:300m"

# Generate embeddings
embeddings = generate_embeddings(model_name, chuncks)

# print("Embeddings Shape:", len(embeddings))
# print("Embeddings type:", type(embeddings))

In [7]:
# storing locally in vectordb (chromadb)
# Using uuid for unique value and avoiding duplicate value to store in chromaDB

import os
import uuid
import chromadb
import numpy as np

# ---------------------------------------------------
# Initialize ChromaDB Vector Store
# ---------------------------------------------------
def initialize_vector_store(collection_name="pdf_documents", persist_directory="../data/vector_store"):
    
    # Create folder if not present
    os.makedirs(persist_directory, exist_ok=True)
    
    # Create persistent ChromaDB client
    client = chromadb.PersistentClient(path=persist_directory)
    client.delete_collection(name=collection_name)

    # Create or load collection
    collection = client.get_or_create_collection(
        name=collection_name,
        metadata={ "description": "PDF document embeddings for RAG" }
    )

    print(f"Collection Name: {collection_name}")
    print(f"Existing Documents: {collection.count()}")

    return collection

# ---------------------------------------------------
# Add Documents + Embeddings
# ---------------------------------------------------
def add_documents_to_vectorstore(collection, documents, embeddings):

    # Empty lists for storage
    ids = []
    metadatas = []
    document_texts = []
    embedding_list = []

    # Loop through each document
    for i, (doc, embedding) in enumerate(zip(documents, embeddings) ):

        # --------------------------------------------
        # Generate Unique ID
        # --------------------------------------------
        doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
        ids.append(doc_id)

        # --------------------------------------------
        # Metadata
        # --------------------------------------------
        metadata = dict(doc.metadata)
        metadata["doc_index"] = i
        metadata["content_length"] = len(doc.page_content)
        metadatas.append(metadata)

        # --------------------------------------------
        # Store document text
        # --------------------------------------------
        document_texts.append(doc.page_content)
        # --------------------------------------------
        # Store embedding
        # --------------------------------------------
        embedding_list.append(embedding[0])

    # ------------------------------------------------
    # Add data into ChromaDB
    # ------------------------------------------------
        
    collection.add(
        ids=ids,
        documents=document_texts,
        metadatas=metadatas,
        embeddings=embedding_list
    )

    print("Documents added successfully")

    print("Total Documents:", collection.count())

collection = initialize_vector_store()
add_documents_to_vectorstore(collection, chuncks, embeddings)   

Collection Name: pdf_documents
Existing Documents: 0
Documents added successfully
Total Documents: 853


In [8]:
# Retriever Pipeline From VectorStore
# Retrieve Similar Documents

def retrieve_documents(query, collection, top_k, score_threshold=0.0):
    
    model_name = "embeddinggemma:300m"
    query_embedding = ollama.embed(model_name, query)
    
    results = collection.query(query_embeddings=query_embedding.embeddings, n_results=top_k)
    
    return results

# query = "what is machine learning?"
# results = retrieve_documents(query, collection, top_k=3)

In [9]:
query="Application of Machine learning in real world"

embedding_result = retrieve_documents(query, collection, top_k=5, score_threshold=0.0)

context = embedding_result['documents']
print(context)
metadata = embedding_result['metadatas'] 
print(metadata)

[['Confusion Matrix\nThe Rare Class Problem\nPrecision, Recall, and Specificity\nROC Curve\nAUC\nLift\nFurther Reading\nStrategies for Imbalanced Data\nUndersampling\nOversampling and Up/Down Weighting\nData Generation\nCost-Based Classification\nExploring the Predictions\nFurther Reading\nSummary\n6. Statistical Machine Learning\nK-Nearest Neighbors\nA Small Example: Predicting Loan Default\nDistance Metrics\nOne Hot Encoder\nStandardization (Normalization, Z-Scores)\nChoosing K', 'Figure 6-9. The predicted outcomes from XGBoost applied to the loan default data', 't-tests, t-Tests-Further Reading\nstatistical inference, classical inference pipeline, Statistical Experiments and\nSignificance Testing\nstatistical machine learning, Statistical Machine Learning-Summary\nbagging and the random forest, Bagging and the Random Forest-\nHyperparameters\nboosting, Boosting-Summary\navoiding overfitting using regularization, Regularization: Avoiding\nOverfitting\nhyperparameters and cross-valida

In [14]:
# Re ranking of the retreive document
from ollama import Client

def rerank_documents(query: str, documents: list[str], top_k: int = 5):
    """Rerank documents using Ollama directly"""
    client = Client()
    scored_docs = []
    
    for doc in documents:
        # BGE reranker expects: [query, document] pairs
        prompt = f"""Given a query and a document, rate the document's relevance to the query.
                    Query: {query}
                    Document: {doc}
                    Relevance Score (0-100):"""
        
        response = client.generate(
            model="qllama/bge-reranker-v2-m3:latest",
            prompt=prompt,
            stream=False,
        )
        
        # Extract score from response
        try:
            score = float(response["response"].strip().split()[0])
        except (ValueError, IndexError):
            score = 0
        
        scored_docs.append((doc, score))
    
    # Return top_k ranked by relevance
    ranked = sorted(scored_docs, key=lambda x: x[1], reverse=True)[:top_k]
    
    return {
        "results": [
            {"document": doc, "score": score}
            for doc, score in ranked
        ]
    }

# Usage in retrieval pipeline
print(rerank_documents(query, embedding_result, top_k=5))

ResponseError: llama-server process no longer running: exit status 0xc0000409: The system detected an overrun of a stack-based buffer in this application. This overrun could potentially allow a malicious user to gain control of this application. GGML_ASSERT(n_outputs_max <= cparams.n_outputs_max) failed (status code: 500)

In [10]:
from ollama import chat

query = "what is K-Means Clustering and it usecase"

embedding_result = retrieve_documents(query, collection, top_k=3, score_threshold=0.0)

context = embedding_result['documents']
# print(context)
metadata = embedding_result['metadatas'] 
# print(metadata)

# -----------------------------------
# Create Prompt
# -----------------------------------
prompt = f"""
You are a helpful AI assistant.

Use ONLY the provided context to answer the question.

Context:
{context}

Question:
{query}

Metadata:
{metadata}

Answer:
"""

# -----------------------------------
# Generate Response from Gemma3
# -----------------------------------
stream = chat(
    model='gemma3:latest',
    messages=[
        {
            'role': 'user',
            'content': prompt
        }
    ],
    stream=True
)

for chunk in stream:
    print(chunk['message']['content'], end='', flush=True)

K-Means Clustering is a technique to divide data into different groups, where the records in each group are similar to one another. A goal of clustering is to identify significant and meaningful groups of data. The groups can be used directly, analyzed in more depth, or passed as a feature or an outcome to a predictive regression or classification model.

A usecase for K-Means Clustering is when a company manages a sales force might want to cluster customers into “personas” to focus and guide sales calls.